In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

MAR, OCT = "March 2026", "October 2026"
GREY = "#a6a6a6"   # March forecast; October takes the theme colour
fy = ["2026-27", "2027-28", "2028-29", "2029-30", "2030-31", "2031-32"]
cy = ["2026", "2027", "2028", "2029", "2030", "2031"]
nan = None  # March forecasts stop a year earlier than October's, so the last row has no March value

M_PSNB = "Public sector net borrowing (% of GDP)"
M_GDP = "GDP growth (%)"
M_CPI = "CPI inflation (%)"

def frame(measure, periods, march, october):
    return pd.DataFrame({"measure": measure, "period": periods,
                         "order": range(len(periods)),
                         "march": march, "october": october}
                        ).astype({"march": float, "october": float})

wide = pd.concat([
    # OBR EFO Table 6.1, % of GDP (financial years)
    frame(M_PSNB, fy, [3.6, 2.9, 2.5, 1.8, 1.6, nan], [4.2, 3.4, 2.9, 2.1, 1.8, 1.8]),
    # Calendar years; October GDP matches the BBC chart and the PA report of the Budget forecasts
    frame(M_GDP, cy, [1.1, 1.6, 1.6, 1.5, 1.5, nan], [1.2, 1.0, 1.6, 1.8, 1.7, 1.6]),
    # Annual average CPI, calendar years
    frame(M_CPI, cy, [2.3, 2.0, 2.0, 2.0, 2.0, nan], [3.1, 3.4, 2.1, 2.1, 2.0, 2.0]),
], ignore_index=True)
measures = list(wide["measure"].unique())

# Pre-formatted text for the hover tooltip and the end label
wide["march_txt"] = wide["march"].map(lambda v: "n/a" if pd.isna(v) else f"{v:.1f}%")
wide["oct_txt"] = wide["october"].map(lambda v: f"{v:.1f}%")
wide["is_last"] = wide["order"] == wide.groupby("measure")["order"].transform("max")
wide["series"] = OCT

# Headroom: an invisible point above each measure's highest value so the y-axis never ends on the data
peak = wide.groupby("measure")[["march", "october"]].transform("max").max(axis=1)
wide["top"] = np.ceil(peak * 1.05 / 0.5) * 0.5

SOURCE = "Source: OBR, Economic and Fiscal Outlook (March and October 2026)"


def build(measure=None):
    """measure=None -> interactive chart with a dropdown; otherwise a static chart for export."""
    interactive = measure is None
    if interactive:
        pick = alt.param(name="selected_measure", value=measures[0],
                         bind=alt.binding_select(options=measures, name="Choose measure: "))
        keep, heading_text = alt.datum.measure == pick, alt.expr("selected_measure")
    else:
        keep, heading_text = alt.datum.measure == measure, measure

    x = alt.X("period:O", sort=alt.EncodingSortField("order", op="min"),
              title=None, axis=alt.Axis(labelAngle=0, ticks=False))
    yscale = alt.Scale(zero=True, nice=True)
    yaxis = alt.Axis(tickCount=5, title=None, labelExpr="format(datum.value, '.1f') + '%'")
    c_oct = alt.Color("series:N", legend=None)   # theme colour for the October series
    tip = [alt.Tooltip("period:N", title="Period"),
           alt.Tooltip("march_txt:N", title=MAR),
           alt.Tooltip("oct_txt:N", title=OCT)]

    def base():
        return alt.Chart(wide).transform_filter(keep)

    headroom = (base().mark_point(opacity=0)
                .encode(x=x, y=alt.Y("top:Q", scale=yscale, axis=yaxis)))
    mar_line = (base().mark_line(strokeWidth=2.4, color=GREY)
                .encode(x=x, y=alt.Y("march:Q", scale=yscale, axis=yaxis)))
    if interactive:
        mar_line = mar_line.add_params(pick)
    oct_line = (base().mark_line(strokeWidth=2.4)
                .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis), color=c_oct))
    mar_pts = (base().mark_point(filled=True, size=70, opacity=1, color=GREY,
                                 stroke="white", strokeWidth=1)
               .encode(x=x, y=alt.Y("march:Q", scale=yscale, axis=yaxis), tooltip=tip))
    oct_pts = (base().mark_point(filled=True, size=70, opacity=1, stroke="white", strokeWidth=1)
               .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis),
                       color=c_oct, tooltip=tip))
    end_lbl = (base().transform_filter(alt.datum.is_last)
               .mark_text(align="left", dx=10, fontSize=12, fontWeight="bold")
               .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis),
                       text="oct_txt:N", color=c_oct))

    # Measure name above the plot (replaces the subtitle; follows the dropdown)
    heading = (alt.Chart(pd.DataFrame({"d": [0]}))
               .mark_text(align="left", fontSize=13, fontWeight="bold", color="#676A86",
                          text=heading_text)
               .encode(x=alt.value(0), y=alt.value(-42)))

    def key_mar(x_px):
        sw = (alt.Chart(pd.DataFrame({"t": [MAR]}))
              .mark_point(filled=True, size=110, opacity=1, color=GREY)
              .encode(x=alt.value(x_px), y=alt.value(-16)))
        tx = (alt.Chart(pd.DataFrame({"t": [MAR]}))
              .mark_text(align="left", dx=9, fontSize=11, color="#676A86")
              .encode(x=alt.value(x_px), y=alt.value(-16), text="t:N"))
        return sw + tx

    def key_oct(x_px):
        sw = (alt.Chart(pd.DataFrame({"series": [OCT]}))
              .mark_point(filled=True, size=110, opacity=1)
              .encode(x=alt.value(x_px), y=alt.value(-16), color=c_oct))
        tx = (alt.Chart(pd.DataFrame({"t": [OCT]}))
              .mark_text(align="left", dx=9, fontSize=11, color="#676A86")
              .encode(x=alt.value(x_px), y=alt.value(-16), text="t:N"))
        return sw + tx

    caption = alt.Title(
        text=SOURCE,
        orient="bottom", anchor="start", fontSize=11, color="#676A86", dy=12)

    return (
        (headroom + mar_line + oct_line + mar_pts + oct_pts + end_lbl + heading
         + key_mar(6) + key_oct(106))
        .properties(width=700, height=270)
        .configure(background="#f7f7f7", font="Circular Std")
        .configure_view(fill="transparent", stroke="transparent")
    )


# Static exports, one per measure: SVG and PNG, plus the Vega-Lite JSON spec
slugs = {M_PSNB: "psnb", M_GDP: "gdp_growth", M_CPI: "cpi_inflation"}
for m in measures:
    c = build(m)
    name = f"obr_revisions_line_{slugs[m]}"
    styles.save(c, path=".", name=name, svg=True)
    c.save(f"{name}.png", scale_factor=2.0)
    c.save(f"{name}.json")

# Interactive version: save its Vega-Lite JSON too (for embedding with the dropdown), then display it
chart = build()
chart.save("obr_revisions_line_dropdown.json")
chart

alt.LayerChart(...)

In [18]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

MAR, OCT = "March 2026", "October 2026"
GREY = "#a6a6a6"   # March forecast; October takes the theme colour
fy = ["2026-27", "2027-28", "2028-29", "2029-30", "2030-31", "2031-32"]
cy = ["2026", "2027", "2028", "2029", "2030", "2031"]
hh_years = ["2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26",
            "2026-27", "2027-28", "2028-29", "2029-30", "2030-31", "2031-32"]
HH_START = "2026-27"   # first year shown for household income; set to "2019-20" to show the full history
nan = None  # March forecasts stop a year earlier than October's, so the last row has no March value

M_PSNB = "Public sector net borrowing (% of GDP)"
M_GDP = "GDP growth (%)"
M_CPI = "CPI inflation (%)"
M_HH = "Real household disposable income per person (% change since 2019-20)"

# OBR EFO Chart 2.6: level of real household disposable income per person (2019-20 = 100)
hh_mar_level = [100.0, 99.6, 99.8, 96.6, 97.4, 100.4, 100.5, 101.0, 101.4, 101.8, 102.2, 102.8, nan]
hh_oct_level = [100.0, 99.6, 99.7, 96.5, 97.3, 100.4, 100.0, 99.4, 99.7, 100.5, 101.0, 101.9, 102.6]
i0 = hh_years.index(HH_START)
hh_periods = hh_years[i0:]
to_pct = lambda levels: [nan if v is None else round(v - 100, 1) for v in levels]   # 0 = 2019-20 level

def frame(measure, periods, march, october):
    return pd.DataFrame({"measure": measure, "period": periods,
                         "order": range(len(periods)),
                         "march": march, "october": october}
                        ).astype({"march": float, "october": float})

wide = pd.concat([
    # OBR EFO Table 6.1, % of GDP (financial years)
    frame(M_PSNB, fy, [3.6, 2.9, 2.5, 1.8, 1.6, nan], [4.2, 3.4, 2.9, 2.1, 1.8, 1.8]),
    # Calendar years; October GDP matches the BBC chart and the PA report of the Budget forecasts
    frame(M_GDP, cy, [1.1, 1.6, 1.6, 1.5, 1.5, nan], [1.2, 1.0, 1.6, 1.8, 1.7, 1.6]),
    # Annual average CPI, calendar years
    frame(M_CPI, cy, [2.3, 2.0, 2.0, 2.0, 2.0, nan], [3.1, 3.4, 2.1, 2.1, 2.0, 2.0]),
    # Household income, financial years, % change since 2019-20
    frame(M_HH, hh_periods, to_pct(hh_mar_level[i0:]), to_pct(hh_oct_level[i0:])),
], ignore_index=True)
measures = list(wide["measure"].unique())

# Pre-formatted text for the hover tooltip and the end labels (household income is signed: +2.6%)
def fmt(m, v):
    if pd.isna(v):
        return "n/a"
    return f"{v:+.1f}%" if m == M_HH else f"{v:.1f}%"

wide["march_txt"] = [fmt(m, v) for m, v in zip(wide["measure"], wide["march"])]
wide["oct_txt"] = [fmt(m, v) for m, v in zip(wide["measure"], wide["october"])]
wide["is_last"] = wide["order"] == wide.groupby("measure")["order"].transform("max")
last_mar = wide.loc[wide["march"].notna()].groupby("measure")["order"].max()
wide["mar_end"] = (wide["measure"] == M_HH) & (wide["order"] == wide["measure"].map(last_mar))  # March label: household income only
wide["series"] = OCT

# Headroom: invisible points above the highest and below the lowest value so the y-axis never ends on the data
peak = wide.groupby("measure")[["march", "october"]].transform("max").max(axis=1)
trough = wide.groupby("measure")[["march", "october"]].transform("min").min(axis=1)
wide["top"] = np.ceil(peak * 1.1 / 0.5) * 0.5
wide["bottom"] = np.floor(np.minimum(trough * 1.1, 0) / 0.5) * 0.5

# "2019-20 level" reference line, household income only
zero_rows = pd.DataFrame({"measure": [M_HH], "period": [hh_periods[-1]], "order": [len(hh_periods) - 1],
                          "y": [0.0], "t": ["2019-20 level"]})

SOURCE = "Source: OBR, Economic and Fiscal Outlook (March and October 2026)"


def build(measure=None):
    """measure=None -> interactive chart with a dropdown; otherwise a static chart for export."""
    interactive = measure is None
    if interactive:
        pick = alt.param(name="selected_measure", value=measures[0],
                         bind=alt.binding_select(options=measures, name="Choose measure: "))
        keep, heading_text, which = alt.datum.measure == pick, alt.expr("selected_measure"), "selected_measure"
    else:
        keep, heading_text, which = alt.datum.measure == measure, measure, repr(measure)

    x = alt.X("period:O", sort=alt.EncodingSortField("order", op="min"),
              title=None, axis=alt.Axis(labelAngle=0, ticks=False, labelOverlap="parity"))
    yscale = alt.Scale(zero=True, nice=True)
    yaxis = alt.Axis(tickCount=6, title=None, labelExpr=(
        f"{which} == '{M_HH}' ? (datum.value > 0 ? '+' : '') + format(datum.value, '.0f') + '%' "
        f": format(datum.value, '.1f') + '%'"))
    c_oct = alt.Color("series:N", legend=None)   # theme colour for the October series
    tip = [alt.Tooltip("period:N", title="Period"),
           alt.Tooltip("march_txt:N", title=MAR),
           alt.Tooltip("oct_txt:N", title=OCT)]

    def base():
        return alt.Chart(wide).transform_filter(keep)

    headroom_top = (base().mark_point(opacity=0)
                    .encode(x=x, y=alt.Y("top:Q", scale=yscale, axis=yaxis)))
    headroom_bottom = (base().mark_point(opacity=0)
                       .encode(x=x, y=alt.Y("bottom:Q", scale=yscale, axis=yaxis)))
    zero = (alt.Chart(zero_rows).transform_filter(keep)
            .mark_rule(color="#94a3b8", strokeDash=[2, 3])
            .encode(y=alt.Y("y:Q", scale=yscale, axis=yaxis)))
    zero_txt = (alt.Chart(zero_rows).transform_filter(keep)
                .mark_text(align="right", dy=12, fontSize=10, color="#64748b")
                .encode(x=x, y=alt.Y("y:Q", scale=yscale, axis=yaxis), text="t:N"))
    mar_line = (base().mark_line(strokeWidth=2.4, color=GREY)
                .encode(x=x, y=alt.Y("march:Q", scale=yscale, axis=yaxis)))
    if interactive:
        mar_line = mar_line.add_params(pick)
    oct_line = (base().mark_line(strokeWidth=2.4)
                .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis), color=c_oct))
    mar_pts = (base().mark_point(filled=True, size=70, opacity=1, color=GREY,
                                 stroke="white", strokeWidth=1)
               .encode(x=x, y=alt.Y("march:Q", scale=yscale, axis=yaxis), tooltip=tip))
    oct_pts = (base().mark_point(filled=True, size=70, opacity=1, stroke="white", strokeWidth=1)
               .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis),
                       color=c_oct, tooltip=tip))
    end_lbl = (base().transform_filter(alt.datum.is_last)
               .mark_text(align="left", dx=10, fontSize=12, fontWeight="bold")
               .encode(x=x, y=alt.Y("october:Q", scale=yscale, axis=yaxis),
                       text="oct_txt:N", color=c_oct))
    mar_end_lbl = (base().transform_filter(alt.datum.mar_end)
                   .mark_text(align="center", dy=-14, fontSize=12, fontWeight="bold", color=GREY)
                   .encode(x=x, y=alt.Y("march:Q", scale=yscale, axis=yaxis), text="march_txt:N"))

    # Measure name above the plot (replaces the subtitle; follows the dropdown)
    heading = (alt.Chart(pd.DataFrame({"d": [0]}))
               .mark_text(align="left", fontSize=13, fontWeight="bold", color="#676A86",
                          text=heading_text)
               .encode(x=alt.value(0), y=alt.value(-42)))

    def key_mar(x_px):
        sw = (alt.Chart(pd.DataFrame({"t": [MAR]}))
              .mark_point(filled=True, size=110, opacity=1, color=GREY)
              .encode(x=alt.value(x_px), y=alt.value(-16)))
        tx = (alt.Chart(pd.DataFrame({"t": [MAR]}))
              .mark_text(align="left", dx=9, fontSize=11, color="#676A86")
              .encode(x=alt.value(x_px), y=alt.value(-16), text="t:N"))
        return sw + tx

    def key_oct(x_px):
        sw = (alt.Chart(pd.DataFrame({"series": [OCT]}))
              .mark_point(filled=True, size=110, opacity=1)
              .encode(x=alt.value(x_px), y=alt.value(-16), color=c_oct))
        tx = (alt.Chart(pd.DataFrame({"t": [OCT]}))
              .mark_text(align="left", dx=9, fontSize=11, color="#676A86")
              .encode(x=alt.value(x_px), y=alt.value(-16), text="t:N"))
        return sw + tx

    caption = alt.Title(
        text=SOURCE,
        orient="bottom", anchor="start", fontSize=11, color="#676A86", dy=12)

    return (
        (headroom_top + headroom_bottom + zero + zero_txt + mar_line + oct_line + mar_pts + oct_pts
         + end_lbl + mar_end_lbl + heading + key_mar(6) + key_oct(106))
        .properties(width=700, height=270)
        .configure(background="#f7f7f7", font="Circular Std")
        .configure_view(fill="transparent", stroke="transparent")
    )


# Static exports, one per measure: SVG and PNG, plus the Vega-Lite JSON spec
slugs = {M_PSNB: "psnb", M_GDP: "gdp_growth", M_CPI: "cpi_inflation", M_HH: "household_income"}
for m in measures:
    c = build(m)
    name = f"obr_revisions_line_{slugs[m]}"
    styles.save(c, path=".", name=name, svg=True)
    c.save(f"{name}.png", scale_factor=2.0)
    c.save(f"{name}.json")

# Interactive version: save its Vega-Lite JSON too (for embedding with the dropdown), then display it
chart = build()
chart.save("obr_revisions_line_dropdown.json")
chart

alt.LayerChart(...)